## Conclusiones

Añadiendo una columna nueva a los datasets de ya pasados por selección de 
características de random forest hemos conseguido en el mejor de los casos pasar de un f1_macro 0.51 a un 0.52.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SEED = 777
np.random.seed(SEED)<


In [ ]:
# cargar el resto de los datos en un diccionario de datasets, donde la clave es el tipo de trato y el valor una tupla (train, test)

datasets = {
    'basic': (pd.read_csv('../data/selected_by_random_forest/data_trainbasic.csv'), pd.read_csv('../data/selected_by_random_forest/data_testbasic.csv')),
    'max_abs': (pd.read_csv('../data/selected_by_random_forest/data_trainmax_abs.csv'), pd.read_csv('../data/selected_by_random_forest/data_testmax_abs.csv')),
    'min_max': (pd.read_csv('../data/selected_by_random_forest/data_trainmin_max.csv'), pd.read_csv('../data/selected_by_random_forest/data_testmin_max.csv')),
    'robust': (pd.read_csv('../data/selected_by_random_forest/data_trainrobust.csv'), pd.read_csv('../data/selected_by_random_forest/data_testrobust.csv')),
    'standard': (pd.read_csv('../data/selected_by_random_forest/data_trainstandard.csv'), pd.read_csv('../data/selected_by_random_forest/data_teststandard.csv'))
}

In [5]:
# Vamos a probar a utilizar isolation forest para asignar una nueva columna
# con una puntuación de rareza

from sklearn.ensemble import IsolationForest

# como se basa en árboles, cojo los datos sin escalar

X_train, X_test, y_train, y_test = datasets['basic']


iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=SEED
)

iso_forest.fit(X_train)


datasets_isolation = {}

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    X_train_iso = X_train.copy()
    X_test_iso = X_test.copy()

    scores_train = iso_forest.decision_function(X_train)
    scores_test = iso_forest.decision_function(X_test)

    X_train_iso['isolation_score'] = scores_train
    X_test_iso['isolation_score'] = scores_test

    datasets_isolation[name] = (X_train_iso, X_test_iso, y_train, y_test)



In [7]:
# a ver si esto cambia algo, empecemos por regresión logística

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

modelos_rl = {}

for nombre, (X_train, X_test, y_train, y_test,) in datasets_isolation.items():
    print(nombre)
    rl = LogisticRegression(max_iter=100000, random_state=SEED, class_weight='balanced')

    rl.fit(X_train, y_train)

    modelos_rl[nombre] = rl

    predicciones = rl.predict(X_test)

    print(confusion_matrix(y_test, predicciones))
    print(classification_report(y_test, predicciones))

basic


/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 12787 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[[20 23]
 [72 91]]
              precision    recall  f1-score   support

           0       0.22      0.47      0.30        43
           1       0.80      0.56      0.66       163

    accuracy                           0.54       206
   macro avg       0.51      0.51      0.48       206
weighted avg       0.68      0.54      0.58       206

max_abs
[[22 21]
 [64 99]]
              precision    recall  f1-score   support

           0       0.26      0.51      0.34        43
           1       0.82      0.61      0.70       163

    accuracy                           0.59       206
   macro avg       0.54      0.56      0.52       206
weighted avg       0.71      0.59      0.62       206

min_max
[[22 21]
 [65 98]]
              precision    recall  f1-score   support

           0       0.25      0.51      0.34        43
           1       0.82      0.60      0.70       163

    accuracy                           0.58       206
   macro avg       0.54      0.56      0.52       206
w

In [9]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# voy a enfocarme en min_max que ha dado mejores resultados

X_train, X_test, y_train, y_test = datasets['min_max']

svc_base = SVC(class_weight='balanced', random_state=SEED)

param_grid_svc = [
    {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10, 50, 100], 
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1] 
    },
    {
        'kernel': ['poly'],
        'degree': [2, 3],           
        'C': [0.1, 1, 10, 50],
        'gamma': ['scale', 'auto']
    }
]

grid_svc = GridSearchCV(
    estimator=svc_base,
    param_grid=param_grid_svc,
    scoring='f1_macro', 
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_svc.fit(X_train, y_train)

print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones))

Fitting 5 folds for each of 46 candidates, totalling 230 fits
[[23 20]
 [65 98]]
              precision    recall  f1-score   support

           0       0.26      0.53      0.35        43
           1       0.83      0.60      0.70       163

    accuracy                           0.59       206
   macro avg       0.55      0.57      0.52       206
weighted avg       0.71      0.59      0.63       206



In [18]:
from sklearn.ensemble import RandomForestClassifier

X_train_basic, X_test_basic, y_train, y_test = datasets_isolation['basic']

rf_model = RandomForestClassifier(
    n_estimators=200,              
    class_weight='balanced',       
    random_state=SEED,
    min_samples_leaf=20,
    n_jobs=-1                      
)

rf_model.fit(X_train_basic, y_train)
y_pred_rf = rf_model.predict(X_test_basic)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


[[ 15  28]
 [ 49 114]]
              precision    recall  f1-score   support

           0       0.23      0.35      0.28        43
           1       0.80      0.70      0.75       163

    accuracy                           0.63       206
   macro avg       0.52      0.52      0.51       206
weighted avg       0.68      0.63      0.65       206



In [ ]:
import xgboost as xgb

xgb_enriquecido = xgb.XGBClassifier(
    random_state=SEED,
    eval_metric='logloss',
    tree_method='hist',          
    scale_pos_weight=0.25        
)

# 3. Entrenamos el modelo con la nueva variable incluida
xgb_enriquecido.fit(X_train_basic, y_train)

# 4. Hacemos las predicciones
y_pred_xgb = xgb_enriquecido.predict(X_test_basic)

# 5. Mostramos la verdad
print("Matriz de Confusión XGBoost Enriquecido:\n", confusion_matrix(y_test, y_pred_xgb))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred_xgb))

Matriz de Confusión XGBoost Enriquecido:
 [[  5  38]
 [ 34 129]]

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.13      0.12      0.12        43
           1       0.77      0.79      0.78       163

    accuracy                           0.65       206
   macro avg       0.45      0.45      0.45       206
weighted avg       0.64      0.65      0.64       206

